# Profitable Wallet Analysis

Loads processed trades from `data/polygon_trades_processed/*.parquet` (top-5%
wallets by training-period P&L). Wallet selection metrics are computed on
**training data only** (`is_train=True`). Final PnL plots cover the full dataset
(training + test).


In [2]:
import math
from pathlib import Path

import pandas as pd
import numpy as np

## Parameters

In [3]:
import datetime

END_DATE_TRAIN = datetime.date(2026, 2, 10)

TRADES_DIR = Path("../data/polygon_trades_processed")


In [4]:
trade_files = sorted(TRADES_DIR.glob("*.parquet"))
if not trade_files:
    raise FileNotFoundError(f"No parquet files found in {TRADES_DIR}")

total_rows = 0
for f in trade_files:
    total_rows += len(pd.read_parquet(f, columns=["wallet"]))

print(f"Found {len(trade_files)} processed shard(s)")
print(f"Total rows across shards: {total_rows:,}")
total_rows


Found 16 processed shard(s)
Total rows across shards: 39,289,062


39289062

## 1 · Load processed trades

In [5]:
df = pd.concat([pd.read_parquet(f) for f in trade_files], ignore_index=True)

# ensure correct dtypes after round-trip
df["dt"] = pd.to_datetime(df["dt"], utc=True)
df["usdc_amount"] = df["usdc_amount"].astype(float)
df["final_value_usdc"] = df["final_value_usdc"].astype(float)
df["quantity"] = df["quantity"].astype(float)
df["position"] = df["position"].astype(float)
df["price"] = df["price"].astype(float)

# backward-compatible aliases for older downstream cells
df["trade_value_usdc"] = df["usdc_amount"]
df["size"] = df["quantity"]
df["transactionHash"] = df["tx_hash"]

# ── PnL per fill ──────────────────────────────────────────────────────────────
df["pnl"] = np.where(
    df["side"] == "BUY",
    df["final_value_usdc"] - df["trade_value_usdc"],
    # SELL: we sold token at price p, implicitly holding the opposite token
    # pnl = trade_value - final_value
    df["trade_value_usdc"] - df["final_value_usdc"],
)

# ── notional per fill ─────────────────────────────────────────────────────────
df["notional"] = np.where(
    df["side"] == "BUY",
    df["trade_value_usdc"],
    df["size"] * (1 - df["price"]),
)

print(f"Loaded {len(df):,} trade records from {len(trade_files):,} shards for {df['wallet'].nunique():,} wallets.")
print(f"  training rows : {df['is_train'].sum():,}")
print(f"  test rows     : {(~df['is_train']).sum():,}")
df.head(3)


Loaded 39,289,062 trade records from 16 shards for 45,407 wallets.
  training rows : 27,380,952
  test rows     : 11,908,110


,wallet,condition_id,dt,side,token_id,outcome,quantity,price,usdc_amount,position,...,trade_pnl,token_winner,final_price,tx_hash,is_train,trade_value_usdc,size,transactionHash,pnl,notional
0,0x126f15c7bd21bf5bf136f3629e10990dc0677137,0x0f416235a6d63a19f2779906242ce173aec3e49bbdcd...,2025-01-01 00:01:05+00:00,BUY,7875320516565813053445635107732149656386289192...,No,10.0,0.62,6.2,10.0,...,3.8,True,1.0,0xdfa6affbb564cebba0f93ad102af9f5b9971270f1c9e...,True,6.2,10.0,0xdfa6affbb564cebba0f93ad102af9f5b9971270f1c9e...,3.8,6.2
1,0x3978a99e028eb590c98d8e9ddbe513298fa92f24,0x0f416235a6d63a19f2779906242ce173aec3e49bbdcd...,2025-01-01 00:01:05+00:00,BUY,5866505527798653441671993940591462145801013733...,Yes,10.0,0.38,3.8,10.0,...,-3.8,False,0.0,0xdfa6affbb564cebba0f93ad102af9f5b9971270f1c9e...,True,3.8,10.0,0xdfa6affbb564cebba0f93ad102af9f5b9971270f1c9e...,-3.8,3.8
2,0x96b59f71f635da5da031e3e93448c54fe226f5e7,0x0f416235a6d63a19f2779906242ce173aec3e49bbdcd...,2025-01-01 00:06:37+00:00,BUY,7875320516565813053445635107732149656386289192...,No,30.0,0.63,18.9,30.0,...,11.1,True,1.0,0xc12324352ff1897431a9cdd5e91bcc05cc7c28e88c38...,True,18.9,30.0,0xc12324352ff1897431a9cdd5e91bcc05cc7c28e88c38...,11.1,18.9


In [6]:
len(df[df['is_train'] == False])

11908110

In [7]:
df.columns

Index(['wallet', 'condition_id', 'dt', 'side', 'token_id', 'outcome',
       'quantity', 'price', 'usdc_amount', 'position', 'final_value_usdc',
       'trade_pnl', 'token_winner', 'final_price', 'tx_hash', 'is_train',
       'trade_value_usdc', 'size', 'transactionHash', 'pnl', 'notional'],
      dtype='object')

In [8]:
# pd.set_option('display.max_rows', None)
# pd.set_option('display.max_colwidth', None)
# df[
#     (df['wallet'] == '0xfe9cf3cacc27249a7573b5777070ee4006f84cb4')
#     & (df['dt'] >= pd.to_datetime("2026-03-07T08:00:00", utc=True))
#     & (df['dt'] <= pd.to_datetime("2026-03-07T11:00:00", utc=True))
#     ][['dt', 'side', 'price', 'quantity', 'pnl', 'usdc_amount', 'wallet', 'token_id', 'condition_id']].head(10)

In [9]:
QUANTILES = [0.001, 0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 0.999, 1]
df[df["is_train"]].groupby("wallet").agg(
    num_trades  = ("tx_hash",       "count"),
    num_markets = ("condition_id",  "nunique"),
    pnl_usdc    = ("pnl",           "sum"),
).describe(percentiles=QUANTILES)


,num_trades,num_markets,pnl_usdc
count,45407.000000,45407.000000,4.540700e+04
mean,603.011694,63.313498,5.474778e+03
std,8415.178701,445.269378,6.534435e+04
min,1.000000,1.000000,-3.302554e+05
0.1%,1.000000,1.000000,-1.265558e+04
1%,1.000000,1.000000,-1.128715e+03
5%,2.000000,1.000000,1.406488e+02
10%,3.000000,1.000000,2.523945e+02
25%,7.000000,3.000000,3.605433e+02
50%,27.000000,9.000000,7.451794e+02


## 2 · Weighted-pnl volatility per wallet (training data only)

Step 1 — collapse fills into **timestamp buckets** `(wallet, dt)` using **training rows only**.  
Step 2 — apply the weighted-return volatility formula across those buckets.

In [10]:
import math
import numpy as np
import pandas as pd

def scaled_weighted_pnl_volatility(buckets: pd.DataFrame) -> float:
    """
    Compute capital-weighted PnL volatility scaled by sqrt(total PnL).

    Each row of `buckets` must contain:
        - notional : total capital in the bucket
        - pnl      : realized PnL in the bucket
    """
    if len(buckets) < 2:
        return float("nan")

    w = buckets["notional"].to_numpy()
    pnl = buckets["pnl"].to_numpy()

    total_w = w.sum()
    total_pnl = pnl.sum()

    if total_w == 0 or total_pnl <= 0:
        return float("nan")

    # weighted mean PnL
    mean = np.sum(w * pnl) / total_w

    # weighted variance
    variance = np.sum(w * (pnl - mean) ** 2) / total_w
    sigma = math.sqrt(variance)

    # scale by sqrt of total PnL
    sigma_scaled = sigma / math.sqrt(total_pnl)

    return sigma_scaled


def _wallet_metrics_from_buckets(group: pd.DataFrame) -> pd.Series:
    """Compute per-wallet metrics from a pre-aggregated bucket DataFrame."""
    pnl = group["pnl"].to_numpy()
    total_notional = group["notional"].sum()
    total_pnl = pnl.sum()

    if total_pnl <= 0:
        top5_pnl_pct = float("nan")
        top_market_pnl_pct = float("nan")
    else:
        top5_pnl = np.sort(pnl)[-5:].sum()
        top5_pnl_pct = top5_pnl / total_pnl
        top_market_pnl_pct = group.groupby("condition_id")["pnl"].sum().max() / total_pnl

    return pd.Series({
        "pnl_volatility":     scaled_weighted_pnl_volatility(group),
        "num_buckets":        len(group),
        "num_markets":        len(group["condition_id"].unique()),
        "total_notional":     total_notional,
        "total_pnl":          total_pnl,
        "top5_pnl_pct":       top5_pnl_pct,
        "top_market_pnl_pct": top_market_pnl_pct,
    })


def compute_wallet_metrics(df_slice: pd.DataFrame) -> pd.DataFrame:
    """
    Given a slice of the fills DataFrame (with columns: wallet, dt, condition_id,
    notional, pnl), aggregate into 5-minute buckets and compute per-wallet metrics.

    Returns a DataFrame indexed by wallet with columns:
        pnl_volatility, num_buckets, num_markets, total_notional,
        total_pnl, top5_pnl_pct, top_market_pnl_pct, return
    """
    tmp = df_slice.copy()
    tmp["dt_floored"] = tmp["dt"].dt.floor("5min")

    buckets = (
        tmp.groupby(["wallet", "dt_floored", "condition_id"], sort=False)
        .agg(notional=("notional", "sum"), pnl=("pnl", "sum"))
        .reset_index()
    )
    buckets = buckets[buckets["notional"] > 0].copy()

    if buckets.empty:
        empty_cols = [
            "wallet", "pnl_volatility", "num_buckets", "num_markets",
            "total_notional", "total_pnl", "top5_pnl_pct", "top_market_pnl_pct", "return",
        ]
        return pd.DataFrame(columns=empty_cols), buckets

    result = (
        buckets.groupby("wallet", sort=False)
        .apply(_wallet_metrics_from_buckets, include_groups=False)
        .reset_index()
    )
    result["return"] = result["total_pnl"] / result["total_notional"]
    return result, buckets

In [11]:
# ── compute metrics on training slice ────────────────────────────────────────
df_train = df[df["is_train"]]
wallet_vol_train, buckets_train = compute_wallet_metrics(df_train)
wallet_vol_train = wallet_vol_train.sort_values("pnl_volatility").reset_index(drop=True)

print(f"Training timestamp buckets: {len(buckets_train):,}  across {buckets_train['wallet'].nunique():,} wallets.")
buckets_train.head(5)

Training timestamp buckets: 11,251,847  across 45,407 wallets.


,wallet,dt_floored,condition_id,notional,pnl
0,0x126f15c7bd21bf5bf136f3629e10990dc0677137,2025-01-01 00:00:00+00:00,0x0f416235a6d63a19f2779906242ce173aec3e49bbdcd...,6.2,3.8
1,0x3978a99e028eb590c98d8e9ddbe513298fa92f24,2025-01-01 00:00:00+00:00,0x0f416235a6d63a19f2779906242ce173aec3e49bbdcd...,3.8,-3.8
2,0x96b59f71f635da5da031e3e93448c54fe226f5e7,2025-01-01 00:05:00+00:00,0x0f416235a6d63a19f2779906242ce173aec3e49bbdcd...,165.1,94.9
3,0xdc2a9a1aedee7c326bd65522890be2a53db697aa,2025-01-01 00:05:00+00:00,0x0f416235a6d63a19f2779906242ce173aec3e49bbdcd...,10.8,-10.8
4,0x3b7b03427034fe97f88e22053595500c7105f74e,2025-01-01 00:15:00+00:00,0x0f416235a6d63a19f2779906242ce173aec3e49bbdcd...,3.2,1.8


In [12]:
WALLET = '0xcceb22d524e186153cffe79f13c0aeb75889f030'

print("Training PnL: " + str(df_train[df_train['wallet'] == WALLET]['pnl'].sum()))

(buckets_train[buckets_train["wallet"] == WALLET]
 .groupby("wallet")
 .apply(scaled_weighted_pnl_volatility, include_groups=False)
 .head(5))

Training PnL: 0.0


,dt_floored,condition_id,notional,pnl
wallet,,,,


In [13]:
df[
    # (df["wallet"] == WALLET) & 
    (df['condition_id'] == '0x680784812145063cecedfa7f000654e89317e2aca612182884d7742ac6f801f9')
    ]

,wallet,condition_id,dt,side,token_id,outcome,quantity,price,usdc_amount,position,...,trade_pnl,token_winner,final_price,tx_hash,is_train,trade_value_usdc,size,transactionHash,pnl,notional
15800484,0xbf216238277f1ba267a7f3bae1a336fd6cb5be19,0x680784812145063cecedfa7f000654e89317e2aca612...,2026-01-03 11:17:49+00:00,BUY,7617928500154880183205143787090187101296671796...,No,1.000000e+01,0.190,1.900000e+00,10.00,...,8.100000e+00,True,1.0,0x985d0b638ce503dc72f6e3bc14222c3f3cb529b9be62...,True,1.900000e+00,1.000000e+01,0x985d0b638ce503dc72f6e3bc14222c3f3cb529b9be62...,8.100000e+00,1.900000e+00
15847771,0x88b0bd7130b184d110c507bce00978e090fbb118,0x680784812145063cecedfa7f000654e89317e2aca612...,2026-01-05 13:40:15+00:00,BUY,7617928500154880183205143787090187101296671796...,No,2.000000e+01,0.180,3.600000e+00,20.00,...,1.640000e+01,True,1.0,0xe5266369cde5bfcc6c92d1a943c9df12a266e9660fe7...,True,3.600000e+00,2.000000e+01,0xe5266369cde5bfcc6c92d1a943c9df12a266e9660fe7...,1.640000e+01,3.600000e+00
15868370,0x88b0bd7130b184d110c507bce00978e090fbb118,0x680784812145063cecedfa7f000654e89317e2aca612...,2026-01-06 05:43:41+00:00,SELL,7617928500154880183205143787090187101296671796...,No,5.550000e+00,0.180,9.990000e-01,14.45,...,-4.551000e+00,True,1.0,0x298bf9a4ee9efe9e507d8efb5c38e88710aff1289320...,True,9.990000e-01,5.550000e+00,0x298bf9a4ee9efe9e507d8efb5c38e88710aff1289320...,-4.551000e+00,4.551000e+00
15868371,0x88b0bd7130b184d110c507bce00978e090fbb118,0x680784812145063cecedfa7f000654e89317e2aca612...,2026-01-06 05:43:41+00:00,SELL,7617928500154880183205143787090187101296671796...,No,1.445000e+01,0.170,2.456500e+00,0.00,...,-1.199350e+01,True,1.0,0x298bf9a4ee9efe9e507d8efb5c38e88710aff1289320...,True,2.456500e+00,1.445000e+01,0x298bf9a4ee9efe9e507d8efb5c38e88710aff1289320...,-1.199350e+01,1.199350e+01
15868372,0x88b0bd7130b184d110c507bce00978e090fbb118,0x680784812145063cecedfa7f000654e89317e2aca612...,2026-01-06 05:43:41+00:00,BUY,6041099785595455628610604883170151776302742194...,Yes,2.855500e+02,0.830,2.370065e+02,285.55,...,-2.370065e+02,False,0.0,0x298bf9a4ee9efe9e507d8efb5c38e88710aff1289320...,True,2.370065e+02,2.855500e+02,0x298bf9a4ee9efe9e507d8efb5c38e88710aff1289320...,-2.370065e+02,2.370065e+02
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16440366,0x751a2b86cab503496efd325c8344e10159349ea1,0x680784812145063cecedfa7f000654e89317e2aca612...,2026-01-30 21:36:36+00:00,BUY,7617928500154880183205143787090187101296671796...,No,3.370000e+02,0.999,3.366630e+02,1137.66,...,3.370000e-01,True,1.0,0xe54a6457cc59755a25a62293faefed316f02c9860396...,True,3.366630e+02,3.370000e+02,0xe54a6457cc59755a25a62293faefed316f02c9860396...,3.370000e-01,3.366630e+02
16440367,0xc8ab97a9089a9ff7e6ef0688e6e591a066946418,0x680784812145063cecedfa7f000654e89317e2aca612...,2026-01-30 21:36:36+00:00,SELL,7617928500154880183205143787090187101296671796...,No,1.421085e-14,0.999,1.419664e-14,0.00,...,-1.421085e-17,True,1.0,0xe54a6457cc59755a25a62293faefed316f02c9860396...,True,1.419664e-14,1.421085e-14,0xe54a6457cc59755a25a62293faefed316f02c9860396...,-1.421085e-17,1.421085e-17
16440368,0xc8ab97a9089a9ff7e6ef0688e6e591a066946418,0x680784812145063cecedfa7f000654e89317e2aca612...,2026-01-30 21:36:36+00:00,BUY,6041099785595455628610604883170151776302742194...,Yes,1.000000e+03,0.001,1.000000e+00,1000.00,...,-1.000000e+00,False,0.0,0xe54a6457cc59755a25a62293faefed316f02c9860396...,True,1.000000e+00,1.000000e+03,0xe54a6457cc59755a25a62293faefed316f02c9860396...,-1.000000e+00,1.000000e+00
16440369,0x751a2b86cab503496efd325c8344e10159349ea1,0x680784812145063cecedfa7f000654e89317e2aca612...,2026-01-30 21:36:36+00:00,BUY,7617928500154880183205143787090187101296671796...,No,1.000000e+03,0.999,9.990000e+02,2137.66,...,1.000000e+00,True,1.0,0xe54a6457cc59755a25a62293faefed316f02c9860396...,True,9.990000e+02,1.000000e+03,0xe54a6457cc59755a25a62293faefed316f02c9860396...,1.000000e+00,9.990000e+02


In [14]:
# ── Step 2: volatility per wallet (training data) ─────────────────────────────
# wallet_vol_train is already computed above via compute_wallet_metrics()
wallet_vol = wallet_vol_train  # alias kept for downstream cells

print(f"Wallets with volatility: {wallet_vol['pnl_volatility'].notna().sum():,}")
wallet_vol.head(10)

Wallets with volatility: 39,706


,wallet,pnl_volatility,num_buckets,num_markets,total_notional,total_pnl,top5_pnl_pct,top_market_pnl_pct,return
0,0xb03400a925627eba43efa653c2ce0ed2bd5ab34e,0.000000e+00,2.0,2.0,2800.000000,2800.000000,1.0,0.500000,1.000000
1,0xc4e584148424c4f5ce8b14ae524b2261a372d522,0.000000e+00,2.0,2.0,1800.000000,1800.000000,1.0,0.500000,1.000000
2,0x14edb8a1e9ccf9c0037856df8010fa5cb7834a1a,0.000000e+00,2.0,2.0,1800.000000,1800.000000,1.0,0.500000,1.000000
3,0x3bffc903c7f8a3db0263d5015b0a9e1f788d5c18,0.000000e+00,2.0,1.0,1200.000000,800.000000,1.0,1.000000,0.666667
4,0x7501f0f257fd7b52cc1dc6be8968e7cab6c9f44b,0.000000e+00,2.0,2.0,2390.000000,2390.000000,1.0,0.500000,1.000000
5,0x1e04e7506b5712fb2a111477fc0a18e8353f721b,0.000000e+00,2.0,1.0,9400.000000,10600.000000,1.0,1.000000,1.127660
6,0xb04b83dc3dcfb9b40b2030f58726603f6dd43ca0,1.261693e-15,2.0,2.0,975.100000,1014.900000,1.0,0.500000,1.040816
7,0x5e88febf1c556f8a6be1ec2dccbd6a42c0ab4089,3.476377e-06,2.0,1.0,17011.890210,249.999790,1.0,1.000000,0.014696
8,0xd20b23fb8e4204b61230d22858ab5f06933e56c2,5.354060e-05,2.0,1.0,1739.994752,1259.996199,1.0,1.000000,0.724138
9,0x3ec488168a820e135412dc6da67c30ee703fd694,4.028256e-03,2.0,2.0,323.999998,886.117301,1.0,0.500139,2.734930


In [15]:
wallet_vol[wallet_vol["wallet"] == WALLET]

,wallet,pnl_volatility,num_buckets,num_markets,total_notional,total_pnl,top5_pnl_pct,top_market_pnl_pct,return


## 3 · Volatility distribution (training)

In [16]:
QUANTILES = [0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
N = wallet_vol["pnl_volatility"].notna().sum()

vol_stats = (
    wallet_vol["pnl_volatility"]
    .dropna()
    .quantile(QUANTILES)
    .rename_axis("quantile")
    .to_frame()
)
vol_stats.insert(0, "wallet_count", [round(q * N) for q in QUANTILES])
vol_stats.loc["mean"] = [float("nan"), wallet_vol["pnl_volatility"].mean()]
vol_stats.loc["std"]  = [float("nan"), wallet_vol["pnl_volatility"].std()]
vol_stats["wallet_count"] = vol_stats["wallet_count"].astype("Int64")
vol_stats["pnl_volatility"] = vol_stats["pnl_volatility"].round(4)

vol_stats

,wallet_count,pnl_volatility
quantile,,
0.01,397,6.433000e-01
0.05,1985,1.581700e+00
0.1,3971,2.320700e+00
0.25,9926,4.247500e+00
0.5,19853,8.599600e+00
0.75,29780,1.666290e+01
0.9,35735,3.147940e+01
0.95,37721,4.892620e+01
0.99,39309,1.345859e+02


In [17]:
len(wallet_vol[(wallet_vol['num_buckets'] >= 20) &
               (wallet_vol['top5_pnl_pct'] <= 0.6)
               & (wallet_vol['return'] >= 0.1)
               ])

568

In [18]:
# ── Select best wallets based on training-period metrics ──────────────────────
filtered_wallets = wallet_vol[
    (wallet_vol['num_buckets'] >= 20) &
    # (wallet_vol['num_buckets'] <= 300) &
    (wallet_vol['top5_pnl_pct'] <= 0.4) &
    (wallet_vol['top_market_pnl_pct'] <= 0.5) 
    # & (wallet_vol['return'] >= 0.05)
    # & (wallet_vol['pnl_volatility'] <= 0.8)
].sort_values("total_pnl", ascending=False)

print(f"Best wallets (selected on training data): {len(filtered_wallets):}")
filtered_wallets.head(20)

Best wallets (selected on training data): 538


,wallet,pnl_volatility,num_buckets,num_markets,total_notional,total_pnl,top5_pnl_pct,top_market_pnl_pct,return
37805,0x6a72f61820b26b1fe4d956e17b6dc2a1ea3033ee,50.431804,18934.0,1746.0,9.071524e+07,7.474621e+06,0.212512,0.126799,0.082397
38361,0xe20a1538293903b746ffe6c4ce2d5c3c0300e469,63.323540,14486.0,4012.0,2.497312e+07,3.165771e+06,0.327568,0.112052,0.126767
18373,0x507e52ef684ca2dd91f90a9d26d149dd3288beae,7.740195,77822.0,16213.0,6.028293e+07,2.397213e+06,0.142453,0.028225,0.039766
9547,0x2005d16a84ceefa912d4e380cd32e7ff827875ea,4.114731,119486.0,15728.0,4.605197e+07,2.148102e+06,0.150436,0.042347,0.046645
34102,0x1bc0d88ca86b9049cf05d642e634836d5ddf4429,25.538579,1285.0,142.0,6.177819e+06,2.137283e+06,0.210785,0.117608,0.345961
37551,0xa9878e59934ab507f9039bcb917c1bae0451141d,46.458006,1971.0,527.0,1.300454e+07,1.573516e+06,0.340465,0.125881,0.120997
33925,0x17db3fcd93ba12d38382a0cade24b200185c5f6d,24.960946,755.0,71.0,4.504210e+06,1.453743e+06,0.277029,0.296902,0.322752
17322,0xee613b3fc183ee44f9da9c05f53e2da107e3debf,7.167593,181462.0,22683.0,8.143650e+07,1.110528e+06,0.293487,0.102315,0.013637
37496,0x1f9b00b2d61b0915e6019d2f2a7bde8888193d2c,45.532396,180.0,8.0,3.217882e+06,1.026989e+06,0.370435,0.471674,0.319151
22589,0xb786b8b6335e77dfad19928313e97753039cb18d,10.288559,30662.0,5296.0,2.374293e+07,9.836435e+05,0.266296,0.067260,0.041429


## 4 · Train vs Test metrics for selected wallets

Re-run `compute_wallet_metrics` on the **test slice** restricted to the selected wallets,
then merge training and test metrics side-by-side for comparison.

In [19]:
best_wallet_set = set(filtered_wallets["wallet"])

# ── test metrics for selected wallets only ────────────────────────────────────
df_test_best = df[~df["is_train"] & df["wallet"].isin(best_wallet_set)]
wallet_vol_test, _ = compute_wallet_metrics(df_test_best)

# ── merge train & test side-by-side ──────────────────────────────────────────
METRIC_COLS = ["wallet", "pnl_volatility", "num_buckets", "num_markets",
               "total_notional", "total_pnl", "return",
               "top5_pnl_pct", "top_market_pnl_pct"]

comparison = (
    filtered_wallets[METRIC_COLS]
    .merge(
        wallet_vol_test[METRIC_COLS],
        on="wallet",
        how="left",
        suffixes=("_train", "_test"),
    )
    .sort_values("total_pnl_train", ascending=False)
    .reset_index(drop=True)
)

print(f"Wallets with test data: {comparison['total_pnl_test'].notna().sum()} / {len(comparison)}")
comparison

Wallets with test data: 415 / 538


,wallet,pnl_volatility_train,num_buckets_train,num_markets_train,total_notional_train,total_pnl_train,return_train,top5_pnl_pct_train,top_market_pnl_pct_train,pnl_volatility_test,num_buckets_test,num_markets_test,total_notional_test,total_pnl_test,return_test,top5_pnl_pct_test,top_market_pnl_pct_test
0,0x6a72f61820b26b1fe4d956e17b6dc2a1ea3033ee,50.431804,18934.0,1746.0,9.071524e+07,7.474621e+06,0.082397,0.212512,0.126799,NaN,2028.0,238.0,9.219732e+06,-5.545559e+04,-0.006015,NaN,NaN
1,0xe20a1538293903b746ffe6c4ce2d5c3c0300e469,63.323540,14486.0,4012.0,2.497312e+07,3.165771e+06,0.126767,0.327568,0.112052,NaN,2097.0,272.0,2.446722e+07,-2.974373e+06,-0.121566,NaN,NaN
2,0x507e52ef684ca2dd91f90a9d26d149dd3288beae,7.740195,77822.0,16213.0,6.028293e+07,2.397213e+06,0.039766,0.142453,0.028225,10.104316,66533.0,12891.0,2.635211e+07,8.717516e+05,0.033081,0.250545,0.089482
3,0x2005d16a84ceefa912d4e380cd32e7ff827875ea,4.114731,119486.0,15728.0,4.605197e+07,2.148102e+06,0.046645,0.150436,0.042347,9.756117,46139.0,7811.0,1.768114e+07,7.227968e+05,0.040880,0.250590,0.062306
4,0x1bc0d88ca86b9049cf05d642e634836d5ddf4429,25.538579,1285.0,142.0,6.177819e+06,2.137283e+06,0.345961,0.210785,0.117608,187.000582,1009.0,181.0,7.344755e+06,1.026210e+05,0.013972,5.017217,1.822424
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
533,0xc835701bc94caf90bf249676e093e367c5c05014,0.125119,104.0,49.0,2.439033e+05,2.441474e+02,0.001001,0.139969,0.029007,0.485622,16.0,3.0,1.951447e+04,1.953400e+01,0.001001,0.873746,0.362291
534,0xc651520d6d3c05cd78c6c5dc98a2304f8097c39f,0.127872,121.0,49.0,2.422490e+05,2.424915e+02,0.001001,0.143530,0.029494,0.120795,3.0,3.0,1.908689e+04,1.910600e+01,0.001001,1.000000,0.371349
535,0x30ef8a3eec7acd7d2922616f70d07785402a5a9e,0.227167,819.0,559.0,2.178886e+04,2.421866e+02,0.011115,0.269386,0.114197,0.223871,1301.0,947.0,4.874464e+04,5.682309e+02,0.011657,0.148921,0.059835
536,0xf5a4e90ed7d1f94979e8281436ecd14b785d1ea3,0.279737,329.0,227.0,1.285989e+03,2.419099e+02,0.188112,0.197181,0.099616,0.817953,36.0,35.0,1.683213e+02,2.167868e+01,0.128793,1.388461,0.318285


In [20]:
comparison[["wallet", "total_pnl_train", "total_pnl_test", "pnl_volatility_train", "pnl_volatility_test", "return_train", "return_test"]].head(20)

,wallet,total_pnl_train,total_pnl_test,pnl_volatility_train,pnl_volatility_test,return_train,return_test
0,0x6a72f61820b26b1fe4d956e17b6dc2a1ea3033ee,7.474621e+06,-5.545559e+04,50.431804,NaN,0.082397,-0.006015
1,0xe20a1538293903b746ffe6c4ce2d5c3c0300e469,3.165771e+06,-2.974373e+06,63.323540,NaN,0.126767,-0.121566
2,0x507e52ef684ca2dd91f90a9d26d149dd3288beae,2.397213e+06,8.717516e+05,7.740195,10.104316,0.039766,0.033081
3,0x2005d16a84ceefa912d4e380cd32e7ff827875ea,2.148102e+06,7.227968e+05,4.114731,9.756117,0.046645,0.040880
4,0x1bc0d88ca86b9049cf05d642e634836d5ddf4429,2.137283e+06,1.026210e+05,25.538579,187.000582,0.345961,0.013972
5,0xa9878e59934ab507f9039bcb917c1bae0451141d,1.573516e+06,NaN,46.458006,NaN,0.120997,NaN
6,0x17db3fcd93ba12d38382a0cade24b200185c5f6d,1.453743e+06,NaN,24.960946,NaN,0.322752,NaN
7,0xee613b3fc183ee44f9da9c05f53e2da107e3debf,1.110528e+06,5.371244e+05,7.167593,7.240645,0.013637,0.011006
8,0x1f9b00b2d61b0915e6019d2f2a7bde8888193d2c,1.026989e+06,NaN,45.532396,NaN,0.319151,NaN
9,0xb786b8b6335e77dfad19928313e97753039cb18d,9.836435e+05,1.532700e+03,10.288559,54.980706,0.041429,0.015554


In [21]:
# comparison['wallet'].tolist()

In [22]:
import plotly.graph_objects as go

# shorten wallet addresses for readability
comparison["wallet_short"] = comparison["wallet"].str[:6] + "..." + comparison["wallet"].str[-4:]

fig = go.Figure([
    go.Bar(
        name="Train PnL",
        x=comparison["wallet_short"],
        y=comparison["total_pnl_train"],
        marker_color="steelblue",
    ),
    go.Bar(
        name="Test PnL",
        x=comparison["wallet_short"],
        y=comparison["total_pnl_test"],
        marker_color="darkorange",
    ),
])
fig.update_layout(
    barmode="group",
    title="Train vs Test PnL per wallet",
    xaxis_title="Wallet",
    yaxis_title="PnL (USDC)",
    xaxis_tickangle=-45,
    legend_title="Period",
)
fig.show(renderer="browser")

In [23]:
fig = go.Figure([
    go.Bar(
        name="Train return",
        x=comparison["wallet_short"],
        y=comparison["return_train"],
        marker_color="steelblue",
    ),
    go.Bar(
        name="Test return",
        x=comparison["wallet_short"],
        y=comparison["return_test"],
        marker_color="darkorange",
    ),
])
fig.update_layout(
    barmode="group",
    title="Train vs Test return (PnL / notional) per wallet",
    xaxis_title="Wallet",
    yaxis_title="Return",
    xaxis_tickangle=-45,
    legend_title="Period",
)
fig.show(renderer="browser")

## 5 · Full-dataset PnL for best wallets

Build timestamp buckets over **all** data (training + test) for the selected wallets,
then plot cumulative PnL with a vertical line marking the train/test split.

In [24]:
import plotly.express as px

# ── full-dataset buckets (train + test) for best wallets ─────────────────────
df_best = df[df["wallet"].isin(best_wallet_set)].copy()
df_best["dt_floored"] = df_best["dt"].dt.floor("1h")
buckets_full = (
    df_best.groupby(["wallet", "dt_floored", "condition_id"], sort=False)
    .agg(notional=("notional", "sum"), pnl=("pnl", "sum"))
    .reset_index()
)
buckets_full = buckets_full[buckets_full["notional"] > 0].copy()

split_line = pd.Timestamp(END_DATE_TRAIN, tz="UTC") + pd.Timedelta(days=1)

print(f"Full-dataset buckets: {len(buckets_full):,}  across {buckets_full['wallet'].nunique():,} wallets.")
print(f"Train/test split at:  {split_line.date()}")

Full-dataset buckets: 2,656,986  across 538 wallets.
Train/test split at:  2026-02-11


In [25]:
# ── per-wallet cumulative PnL (top-20 by training PnL, full dataset) ──────────
top_wallets_plot = filtered_wallets.head(20)["wallet"].tolist()

cumulative_pnl = (
    buckets_full[buckets_full["wallet"].isin(top_wallets_plot)]
    .sort_values(["wallet", "dt_floored"])
    .copy()
)
cumulative_pnl["cumulative_pnl"] = cumulative_pnl.groupby("wallet")["pnl"].cumsum()

fig = px.line(
    cumulative_pnl,
    x="dt_floored",
    y="cumulative_pnl",
    color="wallet",
    title="Cumulative PnL Over Time by Wallet (train + test)",
    labels={"dt_floored": "Time", "cumulative_pnl": "Cumulative PnL (USDC)", "wallet": "Wallet"},
)
fig.add_vline(
    x=split_line.timestamp() * 1000,
    line_dash="dash",
    line_color="black",
    annotation_text=f"Train / Test split ({END_DATE_TRAIN})",
    annotation_position="top left",
)
fig.show(renderer="browser")

In [26]:
# ── combined cumulative PnL across all best wallets ───────────────────────────
cumulative_pnl_all = (
    buckets_full[buckets_full["wallet"].isin(best_wallet_set)]
    .sort_values("dt_floored")
    .copy()
)
cumulative_pnl_all["cumulative_pnl"] = cumulative_pnl_all["pnl"].cumsum()

fig = px.line(
    cumulative_pnl_all,
    x="dt_floored",
    y="cumulative_pnl",
    title="Cumulative PnL Over Time (All Best Wallets, train + test)",
    labels={"dt_floored": "Time", "cumulative_pnl": "Cumulative PnL (USDC)"},
)
fig.add_vline(
    x=split_line.timestamp() * 1000,
    line_dash="dash",
    line_color="black",
    annotation_text=f"Train / Test split ({END_DATE_TRAIN})",
    annotation_position="top left",
)
fig.show(renderer="browser")

In [27]:
print(f"len(df_best): {len(df_best):,}")
df_best[[
    "dt", "condition_id", "outcome", "wallet", "quantity", "position",
    "price", "usdc_amount", "pnl", "is_train", "tx_hash"
]].sort_values("dt").head(10)


len(df_best): 12,095,363


,dt,condition_id,outcome,wallet,quantity,position,price,usdc_amount,pnl,is_train,tx_hash
1,2025-01-01 00:01:05+00:00,0x0f416235a6d63a19f2779906242ce173aec3e49bbdcd...,Yes,0x3978a99e028eb590c98d8e9ddbe513298fa92f24,10.000000,10.000000,0.380,3.800000,-3.800000,True,0xdfa6affbb564cebba0f93ad102af9f5b9971270f1c9e...
24574335,2025-01-01 00:12:09+00:00,0xa0945545bf6ad2d1d7037aa9bf5d1f981f88f7c3eea4...,No,0x033dc6e3e3e0a3ae55402576990392ae910aaf05,190.800000,190.800000,0.938,178.970400,11.829600,True,0x88e75f1918065ad39ab431dbd17317de7d33d4f8432b...
34461814,2025-01-01 00:24:41+00:00,0xe6508d867d153a268bdab732aa8abc8cc57e652d28a2...,No,0x2cde80346eed526242e266476cfcc65073635b98,753.952375,753.952375,0.160,120.632380,633.319995,True,0x17b69e22084ce0ac11af4506b6844a778c3d54c55d06...
27267280,2025-01-01 00:37:23+00:00,0xb09e22aadc33b6747756ff9248475e356adcf3b99fbd...,Yes,0x3978a99e028eb590c98d8e9ddbe513298fa92f24,10.000000,10.000000,0.250,2.500000,-2.500000,True,0x3e62546f847e6ca706a493dd1f2f8eb3584547e96a6e...
7161055,2025-01-01 01:06:53+00:00,0x3b40221adf21a481d1eb3c84bd52e828384fe2558a98...,Yes,0x3a5a19da3731d31c6cfa71449794f8f25b703427,14.500000,40.000000,0.750,10.875000,3.625000,True,0x34b1780ea9a0fb3fd423df489d56e873b773a7284aec...
7161054,2025-01-01 01:06:53+00:00,0x3b40221adf21a481d1eb3c84bd52e828384fe2558a98...,Yes,0x3a5a19da3731d31c6cfa71449794f8f25b703427,25.500000,25.500000,0.750,19.125000,6.375000,True,0x34b1780ea9a0fb3fd423df489d56e873b773a7284aec...
9651261,2025-01-01 01:11:59+00:00,0x4f6be3a3a4d10c128211a0a430e44de4b66e7e9f7ab0...,Yes,0x3978a99e028eb590c98d8e9ddbe513298fa92f24,10.000000,10.000000,0.410,4.100000,-4.100000,True,0x7d6f214bc839214974a12eeb132237e0e2889b89117c...
24574354,2025-01-01 01:17:13+00:00,0xa0945545bf6ad2d1d7037aa9bf5d1f981f88f7c3eea4...,No,0x033dc6e3e3e0a3ae55402576990392ae910aaf05,61.129031,251.929031,0.938,57.339032,3.789999,True,0x58d829f104eca70a6f28ed7789079f252602f48c3648...
12190897,2025-01-01 01:22:59+00:00,0x5a8029020e5886fc3f2fd51f952b39e467c1deb0e4e7...,Yes,0xe97b7b1396a99341b32984fa8bbf62f2f68ab2ea,45.000000,45.000000,0.930,41.850000,3.150000,True,0x796b710279a979e8b83c1bffe7672e90a9e9693b1d29...
27267286,2025-01-01 01:24:35+00:00,0xb8652114aba10ac01aa093e4135a070072a80908f0a0...,No,0x45580feac3b916a88b7513929e26f86b3adea2b3,32.000000,32.000000,0.742,23.744000,8.256000,True,0x89af942ef27855662136832807f5eead7b3928877b8d...


In [28]:
df_best.groupby("condition_id").agg(
    sample_outcome = ("outcome",       "first"),
    total_pnl      = ("pnl",           "sum"),
    num_trades     = ("tx_hash",       "count"),
    total_notional = ("notional",      "sum"),
    wallet_count   = ("wallet",        "nunique"),
).sort_values("total_pnl", ascending=False).head(30)


,sample_outcome,total_pnl,num_trades,total_notional,wallet_count
condition_id,,,,,
0x605c9d91aacb70969836c696d4666c0552b9c123fd72d7f4a2adfd337739ad14,Patriots,593710.223322,13101,1.670739e+06,42
0x9a94d8fa5cb4d339aaac84ab997b8f39975084229fe68fb9888811e3281fbf09,Rockets,492571.237299,2918,9.267796e+05,69
0x204d24f3a0f5dd5fca825292bdeab6a97af3978b2caa2b21bb37e610eddfff5d,Seahawks,488139.301823,13766,3.830183e+06,63
0x3255139811b2eed367c3329581059bd34a37047fd414308c75d7100e7101283b,Seahawks,479886.402084,463,3.542942e+05,21
0x309b383d4c3f4a97d25213570c5f1e795d87ea2161c24aa09d82da8b201e0f25,T1,452279.806163,1043,4.958494e+05,22
0x655e5ca101c466b6293aa15e06173b78b293221803d56e35551f708cd82eb352,Yes,434229.916969,10275,2.255440e+07,98
0x2209d272f45481a4815f9311d94067c2cb2b932546298368f2a3322275169fc6,Saints,425851.265352,575,7.057285e+05,22
0x6b3247fa7243ff3fe24d24b1edee1d69181f08fe305145bd4c59323e9a905324,Warriors,408353.202588,3648,6.541441e+05,60
0x6b24a578e8644f45e49130da20267e466163dc35fee34dd23357a68530d34381,Yes,407923.839748,1414,2.845004e+06,32
